In [8]:
import numpy as np
import matplotlib.pyplot as plt
import torch

In [ ]:
def robust_sigmoid(x: np.ndarray, n_classes: int) -> np.ndarray:
    x = np.abs(x)  # Ensure non-negativity, power related

    # 1. Center the data (Median becomes 0)
    median = np.median(x)
    x_centered = x - median

    # 2. Calculate distribution parameters on the centered scale
    q3 = np.percentile(x_centered, 75)

    # Handle the zero-variance edge case
    if q3 <= 0:
        q3 = 1e-6

    # 3. Sparsity Bias (gamma): Sets p(median) = 1/n
    gamma = np.log(n_classes - 1)

    # 4. Encoding Sensitivity (kappa): Maps centered Q3 to p=0.75
    # Logic: We solve 1 / (1 + exp(-(kappa * q3 - gamma))) = 0.75
    kappa = (gamma + np.log(3)) / q3

    # 5. Transform
    # We use (kappa * x_centered - gamma) so that when x_centered = 0,
    # we are left with -gamma, which gives us our 1/n baseline.
    probabilities = 1 / (1 + np.exp(-(kappa * x_centered - gamma)))

    return probabilities

In [32]:
x = np.random.normal(loc=0, scale=1, size=1000)

x_norm = robust_sigmoid(x, n_classes=10000)

spike_encoded = torch.bernoulli(torch.tensor(x_norm)).float()

spike_rate = spike_encoded.mean().item()
print(f"Spike Rate: {spike_rate:.4f}")

Spike Rate: 0.2750
